# 07d · 오른슈타인-울렌벡 과정 (The Ornstein–Uhlenbeck process: mean reversion, exact solution, stationarity)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Øksendal §5.1 (선형 SDE·Langevin 예제, 연습문제 5.5), §5.2 (존재·유일성) · Shreve II §4.4 (Vasicek 금리모형 예제), §6.5 · G&S 4e §13.3 (확산과정: OU 예제), §9.6 (정상 가우스 과정) · Karlin & Taylor 2e 7장 (보조) |
| 선수 노트북 | 07b (Itô 공식), 07c (오일러–마루야마·강/약수렴), 06b (가우스 과정·공분산 함수) |
| 예상 소요 | 90분 |
| 상태 | draft |

브라운 운동은 어디로든 흘러가고, 기하 브라운 운동은 배수로 흔들리며 한곳으로 되돌아오지 않는다. 그런데 온도·금리·용수철에 매달린 입자의 속도처럼 **어떤 값 주위로 되돌아오는** 잡음 과정이 필요할 때가 많다. 오른슈타인–울렌벡(Ornstein–Uhlenbeck, OU) 과정 $dX=\theta(\mu-X)dt+\sigma dB$ 가 그 표준 모형이다. 이 노트북은 (i) 07b 의 Itô 공식으로 정확해를 구해 전이분포가 가우스임을 완전 증명하고, (ii) 정상분포 $N(\mu,\sigma^2/2\theta)$ 와 자기공분산 $e^{-\theta|\tau|}$ 를 경로 집합(앙상블)과 한 긴 경로(시간평균) 양쪽에서 확인하고, (iii) 07c 에서 배운 EM 의 약오차가 OU 에서는 "정상분산이 $\sigma^2/(2\theta-\theta^2\Delta t)$ 로 부푼다"는 구체적 공식이 됨을 보고, (iv) 한 경로에서 AR(1) 회귀로 모수를 추정해 본다.

## 0. 준비 (Setup)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, compare_table, assert_close
from spkit.sde import ou_exact, euler_maruyama
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("07d")
setup_plots()
N_PATHS = scale(5_000)                        # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_PATHS_EM = scale(20_000, minimum=5_000)    # FAST 에서도 EM 의 약오차가 ≈5 SE 로 잡히도록 하한 5000
T_LONG = float(scale(5_000, minimum=500))     # 한 긴 경로의 길이 (시간 단위)
THETA, MU, SIGMA, X0 = 2.0, 1.0, 0.8, 3.0     # 이 노트북의 기본 모수
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_PATHS_EM={N_PATHS_EM}, T_LONG={T_LONG:g}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다. (06b 의 가우스 과정·공분산 함수 관점은 2절에서 다시 씁니다 — 필요하면 그 노트북의 정리 부분을 잠깐 훑어 보세요.)

**Q1.** (07c) 강수렴(strong convergence)과 약수렴(weak convergence)의 차이는? 각각 EM 의 차수는?

<details><summary>정답</summary>

강: 같은 브라운 경로 위에서 경로별 오차 $E|X_T-X_T^\Delta|$; EM 의 차수는 $1/2$ (가법 잡음이면 1). 약: 분포의 오차 $|Ef(X_T)-Ef(X_T^\Delta)|$; EM 의 차수는 1. 오늘의 OU 는 가법 잡음이라 강차수 1 이지만, 약오차는 여전히 $O(\Delta t)$ 로 남는다 — 그 크기를 정상분산에서 정확히 계산한다.

</details>

**Q2.** (07c) 밀스타인(Milstein) 보정항은 무엇이고, 가법 잡음에서는 왜 사라지는가?

<details><summary>정답</summary>

$\frac12 b\,b_x(\Delta B^2-\Delta t)$. 확산계수 $b$ 가 상수(가법 잡음)이면 $b_x=0$ 이라 보정항이 0 — OU 에서는 밀스타인 = EM 이다.

</details>

**Q3.** (07c) 강오차를 잴 때 왜 같은 브라운 경로를 coarsen 해서 써야 하는가?

<details><summary>정답</summary>

강오차는 경로별로 정의된 양이다. 다른 난수를 쓰면 두 근사가 서로 다른 경로를 따라가 오차가 $O(1)$ 로 남고, $\Delta t$ 를 줄여도 줄지 않는다.

</details>

**Q4.** (07b) $dX=a\,dt+b\,dB$ 에 대한 시간 의존 Itô 공식 ($f(t,x)$) 을 써라.

<details><summary>정답</summary>

$df(t,X_t)=\big(f_t+a\,f_x+\tfrac12b^2f_{xx}\big)dt+b\,f_x\,dB$. 오늘은 $f(t,x)=e^{\theta t}x$ 에 적용한다 — $f_{xx}=0$ 이라 Itô 보정항이 없고 적분인자(integrating factor)만 남는다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **OU 과정.** $dX_t=\theta(\mu-X_t)\,dt+\sigma\,dB_t$, $\theta>0$ (평균회귀 속도, mean-reversion rate), $\mu$ (장기 평균), $\sigma$ (잡음 강도). 물리에서는 Langevin 방정식(브라운 입자의 **속도**), 금융에서는 Vasicek 단기금리 모형 (Shreve II §4.4) 이 바로 이것이다.
- **정확해.** $X_t=\mu+(X_0-\mu)e^{-\theta t}+\sigma\int_0^te^{-\theta(t-s)}dB_s$.
- **전이분포.** $X_t\mid X_0=x\ \sim\ N\!\big(\mu+(x-\mu)e^{-\theta t},\ \tfrac{\sigma^2}{2\theta}(1-e^{-2\theta t})\big)$.
- **정상분포.** $t\to\infty$ 에서 $N(\mu,\sigma^2/2\theta)$. $X_0$ 를 이 분포에서 뽑으면 과정은 (강)정상 가우스 과정이고 $\mathrm{Cov}(X_s,X_t)=\frac{\sigma^2}{2\theta}e^{-\theta|t-s|}$ — 06b 의 언어로, 공분산 함수 $\min(s,t)$ 를 가진 BM 과 나란히 놓이는 두 번째 대표 가우스 과정이다.
- **정확 이산화 (AR(1)).** 격자 $\Delta t$ 위에서 $X_{k+1}=\mu+(X_k-\mu)e^{-\theta\Delta t}+\sqrt{\tfrac{\sigma^2}{2\theta}(1-e^{-2\theta\Delta t})}\;\xi_k$, $\xi_k\sim N(0,1)$ iid. 전이분포 자체를 표본추출하므로 **어떤 $\Delta t$ 에서도 분포가 정확**하다 (`ou_exact`).
- **EM 이산화.** $X_{k+1}=(1-\theta\Delta t)X_k+\theta\mu\Delta t+\sigma\sqrt{\Delta t}\;\xi_k$ — 역시 AR(1) 이지만 계수가 $e^{-\theta\Delta t}$ 대신 그 1차 근사 $1-\theta\Delta t$ 다.

### 2.2 정리 1 — OU 의 정확해와 가우스 전이분포 — Øksendal §5.1, Shreve II §4.4

$X_t=\mu+(X_0-\mu)e^{-\theta t}+\sigma\int_0^te^{-\theta(t-s)}dB_s$ 이고, $X_0=x$ 가 상수이면 $X_t\sim N(m_t,v_t)$,

$$m_t=\mu+(x-\mu)e^{-\theta t},\qquad v_t=\frac{\sigma^2}{2\theta}\big(1-e^{-2\theta t}\big).$$

또한 $s\le t$ 에 대해 $\mathrm{Cov}(X_s,X_t)=e^{-\theta(t-s)}\,v_s=\frac{\sigma^2}{2\theta}e^{-\theta(t-s)}(1-e^{-2\theta s})$.

가정이 왜 필요한가:
- **$\theta>0$**: $e^{-\theta t}\to0$ 이어야 초기값을 잊고 정상분포로 간다. $\theta<0$ 이면 위 공식은 그대로 성립하지만 $v_t$ 가 $e^{2|\theta|t}$ 로 폭발한다 (E3).
- **계수가 $x$ 에 선형**: 적분인자 $e^{\theta t}$ 가 통하고, 해가 $B$ 의 **선형** 범함수가 되어 가우스다. 비선형 드리프트면 해는 가우스가 아니고 닫힌 전이분포도 일반적으로 없다.
- **결정론적 피적분함수 $e^{-\theta(t-s)}$**: 가우스성과 등거리성(Itô isometry)에 의한 분산 계산 $\sigma^2\int_0^te^{-2\theta(t-s)}ds$ 가 정확히 되는 이유. 피적분함수가 확률적이면 등거리성은 여전히 성립하지만 결과가 기댓값 형태로만 남는다.

<details><summary>증명</summary>

**1. 적분인자.** $f(t,x)=e^{\theta t}x$ 에 시간 의존 Itô 공식(07b, Q4)을 적용한다. $f_t=\theta e^{\theta t}x$, $f_x=e^{\theta t}$, $f_{xx}=0$, 그리고 $a=\theta(\mu-x)$, $b=\sigma$ 이므로

$$d\big(e^{\theta t}X_t\big)=e^{\theta t}\big[\theta X_t+\theta(\mu-X_t)\big]dt+e^{\theta t}\sigma\,dB_t=\theta\mu e^{\theta t}dt+\sigma e^{\theta t}dB_t.$$

$X_t$ 가 들어 있는 항이 정확히 상쇄된다 — 이것이 "선형 SDE 는 적분인자로 푼다"는 말의 내용이다. $0$ 부터 $t$ 까지 적분하면

$$e^{\theta t}X_t=X_0+\mu\big(e^{\theta t}-1\big)+\sigma\int_0^te^{\theta s}dB_s,$$

양변에 $e^{-\theta t}$ 를 곱해 정확해를 얻는다. (해의 유일성은 계수가 립시츠라 Øksendal §5.2 의 정리로 보장된다.)

**2. 가우스성.** $I_t:=\int_0^te^{-\theta(t-s)}dB_s$ 의 피적분함수는 결정론적 연속 함수다. 07a 의 구성대로 $I_t$ 는 단순 과정 근사 $\sum_ie^{-\theta(t-t_i)}\Delta B_i$ 의 $L^2$ 극한이고, 각 근사는 독립 가우스 증분의 **선형결합**이라 가우스다. 가우스 확률변수의 $L^2$ 극한은 가우스(특성함수가 수렴)이므로 $I_t$ 는 가우스이고, $X_0=x$ 상수이면 $X_t=m_t+\sigma I_t$ 도 가우스다.

**3. 평균.** Itô 적분의 평균은 0 (07a 마팅게일 성질) 이므로 $E[X_t]=\mu+(x-\mu)e^{-\theta t}=m_t$.

**4. 분산 (등거리성).** $\mathrm{Var}(X_t)=\sigma^2E[I_t^2]=\sigma^2\int_0^te^{-2\theta(t-s)}ds=\sigma^2\Big[\frac{e^{-2\theta(t-s)}}{2\theta}\Big]_{s=0}^{s=t}=\frac{\sigma^2}{2\theta}\big(1-e^{-2\theta t}\big)=v_t.$

**5. 공분산.** $s\le t$ 에 대해 정확해를 $s$ 에서 다시 시작하면 (마르코프 성질; 적분을 $[0,s]$ 와 $[s,t]$ 로 나누면 바로 나온다)

$$X_t-m_t=e^{-\theta(t-s)}\big(X_s-m_s\big)+\sigma\int_s^te^{-\theta(t-u)}dB_u.$$

둘째 항은 $[s,t]$ 의 브라운 증분만으로 만들어져 $\mathcal F_s$ (따라서 $X_s$) 와 독립이고 평균 0 이다. 그러므로 $\mathrm{Cov}(X_s,X_t)=e^{-\theta(t-s)}\mathrm{Var}(X_s)=e^{-\theta(t-s)}v_s$. $\blacksquare$

</details>

### 2.3 정리 2 — 정상분포와 자기공분산, 에르고딕성 — G&S 4e §9.6, §13.3

$\theta>0$ 이면 어떤 $X_0$ 에서 출발해도 $X_t\Rightarrow N(\mu,\sigma^2/2\theta)$. $X_0\sim N(\mu,\sigma^2/2\theta)$ 로 시작하면 과정은 정상이고 $\mathrm{Cov}(X_s,X_t)=\frac{\sigma^2}{2\theta}e^{-\theta|t-s|}$. 또한 에르고딕(ergodic)이다:

$$\frac1T\int_0^TX_t\,dt\to\mu,\qquad\frac1T\int_0^T(X_t-\mu)^2dt\to\frac{\sigma^2}{2\theta}\quad\text{a.s.},\qquad \mathrm{Var}\Big(\frac1T\int_0^TX_t\,dt\Big)\approx\frac{2}{\theta T}\cdot\frac{\sigma^2}{2\theta}=\frac{\sigma^2}{\theta^2T}.$$

가정이 왜 필요한가:
- **$\theta>0$**: 정리 1 과 같다.
- **정상 초기분포**: 공분산이 $|t-s|$ 에만 의존하려면 $v_s\equiv\sigma^2/2\theta$ 여야 한다. 상수 $X_0$ 에서 출발하면 $v_s<\sigma^2/2\theta$ 라 공분산이 $s$ 에도 의존한다 (E1) — 다만 $s\gg1/\theta$ 면 차이는 $e^{-2\theta s}$ 로 무시할 만하다.
- **자기공분산의 적분가능성** $\int_0^\infty e^{-\theta\tau}d\tau=1/\theta<\infty$: 시간평균의 분산이 $O(1/T)$ 가 되는 조건이고, batch means 로 SE 를 추정하는 근거다.

*증명 스케치.* 정리 1 에서 $t\to\infty$: $m_t\to\mu$, $v_t\to\sigma^2/2\theta$, 가우스 분포는 평균·분산으로 결정되므로 분포수렴. 정상 초기값이면 $v_s\equiv\sigma^2/2\theta$ 이므로 정리 1 의 공분산 공식이 $\frac{\sigma^2}{2\theta}e^{-\theta|t-s|}$ 가 된다. 시간평균의 분산:

$$\mathrm{Var}\Big(\frac1T\int_0^TX\Big)=\frac1{T^2}\int_0^T\!\!\int_0^T\mathrm{Cov}(X_s,X_t)\,ds\,dt=\frac{2}{T^2}\int_0^T(T-\tau)\frac{\sigma^2}{2\theta}e^{-\theta\tau}d\tau\approx\frac2T\int_0^\infty\frac{\sigma^2}{2\theta}e^{-\theta\tau}d\tau=\frac{\sigma^2}{\theta^2T}.$$

즉 길이 $T$ 의 경로는 독립 표본 $n_{\mathrm{eff}}=\theta T/2$ 개만큼의 정보를 준다 (정상분산 $\sigma^2/2\theta$ 를 $n_{\mathrm{eff}}$ 로 나눈 것). 에르고딕 정리 자체(a.s. 수렴)는 정상 가우스 과정의 일반 결과로 G&S 4e §9.6 을 인용한다. $\square$

### 2.4 정리 3 — EM 이산화의 정상분산

$0<\theta\Delta t<2$ 이면 EM 사슬 $X_{k+1}=(1-\theta\Delta t)X_k+\theta\mu\Delta t+\sigma\sqrt{\Delta t}\,\xi_k$ 는 정상분포

$$N\Big(\mu,\ \frac{\sigma^2}{2\theta-\theta^2\Delta t}\Big)$$

를 가지며, 정상분산은 참값 $\frac{\sigma^2}{2\theta}$ 보다 $\frac{\sigma^2}{2\theta}\cdot\frac{\theta\Delta t/2}{1-\theta\Delta t/2}$ 만큼 크다 (약오차 $O(\Delta t)$). 평균은 정확히 $\mu$ 다. $\theta\Delta t\ge2$ 이면 분산이 발산한다. 반면 정확 이산화(AR(1) 계수 $e^{-\theta\Delta t}$)는 모든 $\Delta t$ 에서 정상분산이 정확히 $\sigma^2/2\theta$ 다.

가정이 왜 필요한가:
- **$|1-\theta\Delta t|<1$**: AR(1) $Y_{k+1}=aY_k+\varepsilon_k$ 가 정상해를 가지려면 $|a|<1$ 이어야 한다. 정확 이산화의 계수 $e^{-\theta\Delta t}$ 는 항상 $(0,1)$ 이라 조건이 없다. $1<\theta\Delta t<2$ 에서는 $a<0$ — 수렴은 하지만 경로가 매 스텝 $\mu$ 를 가로지르며 진동한다 (E2 에서 $\theta\Delta t=1$, 즉 $a=0$ 인 경계를 본다).

<details><summary>증명</summary>

$Y_k=X_k-\mu$ 라 두면 EM 재귀는 $Y_{k+1}=aY_k+\sigma\sqrt{\Delta t}\,\xi_k$, $a=1-\theta\Delta t$. 평균: $E[Y_{k+1}]=aE[Y_k]$ 이므로 $E[Y_k]=a^kY_0\to0$, 즉 $E[X_k]\to\mu$ — 참 과정의 $e^{-\theta k\Delta t}$ 대신 $a^k$ 로 감쇠할 뿐 극한은 같다. 분산: $\xi_k$ 가 $Y_k$ 와 독립이므로 $\mathrm{Var}(Y_{k+1})=a^2\mathrm{Var}(Y_k)+\sigma^2\Delta t$. 이 1차 재귀는 $|a|<1$ 일 때 유일한 고정점 $v$ 로 수렴하고, $v=a^2v+\sigma^2\Delta t$ 에서

$$v=\frac{\sigma^2\Delta t}{1-(1-\theta\Delta t)^2}=\frac{\sigma^2\Delta t}{2\theta\Delta t-\theta^2\Delta t^2}=\frac{\sigma^2}{2\theta-\theta^2\Delta t}.$$

가우스성: $Y_k$ 는 가우스 $\xi$ 들의 선형결합이므로 각 $k$ 에서 가우스이고, 극한도 가우스. 초과분: $\frac{\sigma^2}{2\theta-\theta^2\Delta t}-\frac{\sigma^2}{2\theta}=\frac{\sigma^2}{2\theta}\Big(\frac{1}{1-\theta\Delta t/2}-1\Big)=\frac{\sigma^2}{2\theta}\cdot\frac{\theta\Delta t/2}{1-\theta\Delta t/2}$. $|a|\ge1$ 이면 $\mathrm{Var}(Y_k)=\sigma^2\Delta t\sum_{j<k}a^{2j}\to\infty$.

정확 이산화: $a=e^{-\theta\Delta t}$, 잡음 분산 $\frac{\sigma^2}{2\theta}(1-e^{-2\theta\Delta t})$ 이므로 고정점 방정식 $v=e^{-2\theta\Delta t}v+\frac{\sigma^2}{2\theta}(1-e^{-2\theta\Delta t})$ 의 해는 $v=\frac{\sigma^2}{2\theta}$ — $\Delta t$ 가 사라진다. 예: $\theta=2,\Delta t=0.1$: EM $0.64/3.6=0.1778$ vs 정확 $0.16$ (11 % 초과). $\blacksquare$

</details>

### 2.5 직관

OU 는 "용수철에 매달린 브라운 입자"다: $-\theta(X-\mu)$ 가 평균 쪽으로 끌어당기고 $\sigma dB$ 가 흔든다. 두 힘이 균형을 이루는 폭이 정상 표준편차 $\sigma/\sqrt{2\theta}$ 다 — 분모의 $2$ 는 등거리성 $\int_0^\infty e^{-2\theta s}ds=1/(2\theta)$ 에서 온다 (지수 안의 $2\theta$ 가 분모로 내려온다). 기억은 $e^{-\theta\tau}$ 로 사라지므로 상관시간(correlation time)은 $1/\theta$; 긴 경로 $T$ 는 대략 $\theta T/2$ 개의 독립 표본만큼의 정보만 준다.

이산화하면 OU 는 정확히 AR(1) 이 되어 (i) 회귀로 $\theta,\mu,\sigma$ 를 추정할 수 있고, (ii) EM 은 계수 $e^{-\theta\Delta t}$ 를 $1-\theta\Delta t$ 로 바꾼 "틀린 AR(1)" 임이 눈에 보인다 — 07c 의 추상적인 약오차 $O(\Delta t)$ 가 여기서는 $\sigma^2/(2\theta-\theta^2\Delta t)$ 라는 구체적 숫자다. 06b 의 가우스 과정 관점(공분산 함수가 과정을 규정)이 다시 쓰인다: BM 은 $\min(s,t)$, OU 는 $\frac{\sigma^2}{2\theta}e^{-\theta|s-t|}$.

### 2.6 함정

1. 정상분산은 $\sigma^2/\theta$ 가 **아니라** $\sigma^2/(2\theta)$ 다.
2. "정상"과 "평균회귀"는 같은 말이 아니다: $\theta>0$ 이 필요하고, $\theta<0$ 이면 같은 공식이 분산 $\propto e^{2|\theta|t}$ 를 준다 (E3).
3. 한 긴 경로의 시간평균 SE 는 $s/\sqrt n$ 이 아니다 — 표본이 상관되어 있어 유효 표본 수는 $\approx\theta T/2$. batch means 로 추정한다.
4. EM 은 $\theta\Delta t\ge2$ 에서 발산하지만 정확 AR(1) 이산화는 아무 $\Delta t$ 에서도 안정 — 선형 SDE 에는 EM 을 쓸 이유가 없다.
5. AR(1) 회귀 추정량 $\hat\theta=-\log\hat a/\Delta t$ 의 SE 는 ($T$ 고정) $\Delta t$ 를 줄여도 거의 줄지 않는다: $\mathrm{se}(\hat\theta)\approx\sqrt{(1-a^2)/n}/(a\Delta t)\approx\sqrt{2\theta/T}$. 정보는 관측 수가 아니라 $\theta T$ 에 비례한다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — OU 의 정상분산에 왜 $2\theta$ 가 나오는지 등거리성 적분 한 줄로 설명하고, 정확 이산화와 EM 이 AR(1) 계수에서 어떻게 다른지 1문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

기본 모수는 $\theta=2,\ \mu=1,\ \sigma=0.8,\ X_0=3$ 입니다. 아래 값을 **손으로** 계산해 적어 보세요. 시뮬레이션 뒤 5절의 표에서 자기 예측과 비교합니다.

In [ ]:
predictions = {
    # E[X_{0.5}] 은? (정리 1 의 m_t)
    "mean_t05": None,
    # Var(X_{0.5}) 은? (정리 1 의 v_t)
    "var_t05": None,
    # 정상분산 σ²/(2θ) 은? (t=5 에서는 사실상 정상 상태)
    "var_stat": None,
    # 정상 상태에서 Cov(X_{4.5}, X_5) 은? (정리 2 의 자기공분산)
    "acov_lag05": None,
    # EM (Δt=0.1) 으로 t=5 까지 돌렸을 때 Var(X_5^EM) 은? (정리 3; 정상분산 참값과 다르다)
    "em_var_dt01": None,
    # 한 긴 경로 (T=T_LONG: full 5000 / FAST 500, Δt=0.05) 의 AR(1) 회귀에서 θ̂ = -log â/Δt 는 대략? (참값 2; SE 도 어림해 보세요 — 함정 5)
    "theta_hat": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서 반복해 쓰는 작은 통계 헬퍼를 정의한다: 표본분산의 SE, 상관된 시계열의 batch-means SE, 경험 자기공분산, AR(1) 회귀.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def var_se(x):
    """표본분산 s² 의 SE (정규 근사): s²·sqrt(2/(n-1))."""
    s2 = x.var(ddof=1)
    return s2 * np.sqrt(2.0 / (len(x) - 1))

def batch_means_se(x, n_batches=50):
    """상관된 시계열의 시간평균 SE — n_batches 개 배치 평균의 산포로 추정."""
    b = x[: len(x) // n_batches * n_batches].reshape(n_batches, -1).mean(1)
    return b.std(ddof=1) / np.sqrt(n_batches)

def acov(x, lag_steps):
    """경험 자기공분산 (표본평균으로 중심화)."""
    m = x.mean()
    return np.mean((x[: len(x) - lag_steps] - m) * (x[lag_steps:] - m))

def ar1_fit(x, dt):
    """AR(1) 회귀 x_{k+1} = a x_k + b + ε 에서 (θ̂, μ̂, σ̂, se_θ) 를 돌려준다."""
    a, b = np.polyfit(x[:-1], x[1:], 1)
    resid = x[1:] - (a * x[:-1] + b)
    s2 = resid.var(ddof=2)
    theta = -np.log(a) / dt
    mu = b / (1 - a)
    sigma = np.sqrt(2 * theta * s2 / (1 - a**2))       # 잡음분산 = σ²(1-a²)/(2θ) 를 뒤집은 것
    se_theta = np.sqrt((1 - a**2) / (len(x) - 1)) / (a * dt)   # delta method: se(â)/(â Δt)
    return theta, mu, sigma, se_theta

### Step 1 — 경로, 평균 곡선, $\pm2$ 표준편차 띠

`ou_exact` 는 2.1 의 정확 이산화(AR(1)) 로 경로를 만든다. 경로 10개 위에 정리 1 의 $m_t$, $m_t\pm2\sqrt{v_t}$ 를 겹치고, 오른쪽에는 모든 경로의 앙상블 평균·분산 곡선을 $m_t, v_t$ 와 비교한다.

In [ ]:
def ou_mean(t, x0=X0):
    return MU + (x0 - MU) * np.exp(-THETA * t)

def ou_var(t):
    return SIGMA**2 / (2 * THETA) * (1 - np.exp(-2 * THETA * t))

V_STAT = SIGMA**2 / (2 * THETA)          # 정상분산 σ²/2θ
T, N_STEPS = 5.0, 500
t, X = ou_exact(THETA, MU, SIGMA, X0, T, N_STEPS, rng, N_PATHS)
DT = t[1] - t[0]
m_t, v_t = ou_mean(t), ou_var(t)
print(f"X shape {X.shape}, Δt = {DT}, stationary sd = {np.sqrt(V_STAT):.3f}, correlation time 1/θ = {1 / THETA}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
plot_paths(t, X, ax=axes[0], n_show=10, alpha=0.5, lw=0.8)
axes[0].plot(t, m_t, "k", lw=2, label="$m_t=\\mu+(x_0-\\mu)e^{-\\theta t}$")
axes[0].fill_between(t, m_t - 2 * np.sqrt(v_t), m_t + 2 * np.sqrt(v_t), color="k", alpha=0.12,
                     label="$m_t\\pm2\\sqrt{v_t}$")
axes[0].axhline(MU, color="C3", ls="--", lw=1, label="$\\mu$")
axes[0].set(ylabel="$X_t$", title=f"OU paths from $x_0$={X0:g} (θ={THETA:g}, μ={MU:g}, σ={SIGMA:g})")
axes[0].legend(loc="upper right")
axes[1].plot(t, X.mean(0), label="ensemble mean")
axes[1].plot(t, m_t, "k--", label="$m_t$")
axes[1].plot(t, X.var(0, ddof=1), label="ensemble variance")
axes[1].plot(t, v_t, "k:", label="$v_t$")
axes[1].axhline(V_STAT, color="C3", ls="--", lw=1, label="$\\sigma^2/2\\theta$")
axes[1].set(xlabel="t", title=f"Ensemble mean / variance over {N_PATHS} paths vs $m_t$, $v_t$")
axes[1].legend(loc="upper right")
plt.show()

경로들은 $\approx 1/\theta=0.5$ 시간 안에 $3$ 을 잊고 $\mu=1$ 주위 폭 $\pm2\times0.4$ 의 띠에 자리잡는다. 오른쪽 패널에서 분산 곡선이 $0$ 에서 출발해 $\sigma^2/2\theta=0.16$ 으로 **포화**하는 것이 BM 과의 결정적 차이다 — BM 의 분산은 $t$ 로 계속 커진다.

### Step 2 — 전이분포 $X_{0.5}\mid X_0=3$

정리 1 의 $N(m_{0.5},v_{0.5})$ 와 히스토그램을 겹치고, 평균은 `mc_estimate`, 분산은 표본분산 (SE $=s^2\sqrt{2/(n-1)}$) 으로 잰다.

In [ ]:
k05 = int(round(0.5 / DT))
x05 = X[:, k05]
m05, v05 = ou_mean(t[k05]), ou_var(t[k05])
grid = np.linspace(x05.min(), x05.max(), 300)
ax = plot_hist_vs_pdf(x05, grid, norm.pdf(grid, m05, np.sqrt(v05)),
                      label_pdf=f"N({m05:.4f}, {v05:.4f})")
ax.set(xlabel="$X_{0.5}$", title="$X_{0.5}\\,|\\,X_0=3$: histogram vs exact Gaussian transition")
plt.show()
print("mean X_0.5 :", mc_estimate(x05), f" vs m_0.5 = {m05:.6f}")
print(f"var  X_0.5 : {x05.var(ddof=1):.5f} ± {var_se(x05):.5f}  vs v_0.5 = {v05:.6f}")

### Step 3 — 정상분포 (앙상블, $t=5$) 와 lag-0.5 자기공분산

$t=5$ 는 상관시간의 10 배라 전이분포와 정상분포의 차이 ($m_5-\mu=2e^{-10}$, $1-v_5/v_\infty=e^{-20}$) 는 SE 보다 훨씬 작다. 자기공분산은 참값 $\mu$ 를 알고 중심화한 곱 $(X_{4.5}-\mu)(X_5-\mu)$ 의 **평균**으로 추정하면 SE 가 `mc_estimate` 에서 바로 나온다.

In [ ]:
x5, x45 = X[:, -1], X[:, int(round(4.5 / DT))]
m5, v5 = ou_mean(T), ou_var(T)
rho05 = np.exp(-THETA * 0.5)                     # lag 0.5 의 상관계수 e^{-θτ}
print(f"transient left at t=5: m_5 - μ = {m5 - MU:.1e}, 1 - v_5/v_stat = {1 - v5 / V_STAT:.1e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
grid = np.linspace(x5.min(), x5.max(), 300)
plot_hist_vs_pdf(x5, grid, norm.pdf(grid, MU, np.sqrt(V_STAT)), ax=axes[0],
                 label_pdf="stationary N(μ, σ²/2θ)")
axes[0].set(xlabel="$X_5$", title=f"$X_5$: histogram vs stationary N({MU:g}, {V_STAT:g})")
axes[1].plot(x45, x5, ".", ms=3, alpha=0.4)
xx = np.array([x45.min(), x45.max()])
axes[1].plot(xx, MU + rho05 * (xx - MU), "C1", lw=2, label=f"regression line, slope $e^{{-\\theta/2}}$ = {rho05:.3f}")
axes[1].set(xlabel="$X_{4.5}$", ylabel="$X_5$", title="Lag-0.5 scatter in the stationary regime")
axes[1].legend()
plt.show()

In [ ]:
acov05 = mc_estimate((x45 - MU) * (x5 - MU))
print("mean X_5       :", mc_estimate(x5), f" vs μ = {MU}")
print(f"var  X_5       : {x5.var(ddof=1):.5f} ± {var_se(x5):.5f}  vs σ²/2θ = {V_STAT}")
print("Cov(X_4.5, X_5):", acov05, f" vs (σ²/2θ) e^(-θ·0.5) = {V_STAT * rho05:.6f}")
print(f"sample correlation {np.corrcoef(x45, x5)[0, 1]:.4f}  vs e^(-θ·0.5) = {rho05:.4f}")

산점도의 회귀직선 기울기 $e^{-\theta\cdot0.5}=0.368$ 은 $\Delta t=0.5$ 격자의 정확 이산화 AR(1) 계수다 (이 노트북의 $\Delta t=0.01$ 격자에서는 계수 $e^{-0.02}$ 를 50 번 곱한 값) — 0.5 시간 뒤에는 편차의 37 % 만 남는다.

### Step 4 — 에르고딕: 한 긴 경로의 시간평균

앙상블 대신 **경로 하나**를 $T$ 시간 동안 따라가며 시간평균을 낸다. 표본이 상관되어 있으므로 SE 는 batch means 로: 경로를 50 배치로 잘라 (배치 길이 $T/50\gg$ 상관시간 $0.5$) 배치 평균의 산포를 쓴다. 경로는 $X_0=3$ 에서 출발하고 burn-in 을 따로 버리지 않는다 — 과도기가 시간평균에 남기는 결정론적 편향은 평균에 $\frac1T\int_0^\infty(x_0-\mu)e^{-\theta t}dt=\frac{x_0-\mu}{\theta T}$, 분산에 $\frac{(x_0-\mu)^2}{2\theta T}$ 로 (여기서는 둘 다 $1/T$), FAST 의 $T=500$ 에서도 $0.002$ 라 SE 보다 한참 작다 (코드에서 확인).

In [ ]:
DT_LONG, N_BATCH = 0.05, 50
n_long = int(round(T_LONG / DT_LONG))
_, Xl = ou_exact(THETA, MU, SIGMA, X0, T_LONG, n_long, rng, n_paths=1)
Xl = Xl[0, 1:]                                   # 상수 X_0 = 3 은 빼고 n_long 개 관측
tavg_mean = Xl.mean()
tavg_var = ((Xl - tavg_mean) ** 2).mean()
se_mean = batch_means_se(Xl, N_BATCH)
se_var = batch_means_se((Xl - tavg_mean) ** 2, N_BATCH)
se_theory = np.sqrt(SIGMA**2 / (THETA**2 * T_LONG))       # sqrt(2 v_stat / (θT))  (정리 2)
se_naive = Xl.std(ddof=1) / np.sqrt(n_long)               # 독립 표본인 척한 SE (틀림)
print(f"one path: T = {T_LONG:g}, Δt = {DT_LONG}, n = {n_long}, batches = {N_BATCH} x {n_long // N_BATCH} points "
      f"({n_long // N_BATCH * DT_LONG:g} time units each)")
print(f"time-average mean = {tavg_mean:.4f} ± {se_mean:.4f} (batch means)  vs μ = {MU};   theory SE σ/(θ√T) = {se_theory:.4f}")
print(f"time-average var  = {tavg_var:.4f} ± {se_var:.4f} (batch means)  vs σ²/2θ = {V_STAT}")
print(f"naive SE s/√n = {se_naive:.4f} is too small by {se_theory / se_naive:.1f}x;  n_eff = θT/2 = {THETA * T_LONG / 2:.0f} of n = {n_long}")
bias_mean, bias_var = (X0 - MU) / (THETA * T_LONG), (X0 - MU) ** 2 / (2 * THETA * T_LONG)   # burn-in 을 안 버린 대가
print(f"transient bias from X_0 = {X0:g} (no burn-in): mean {bias_mean:.4f}, var {bias_var:.4f}  — both ≪ SE")

In [ ]:
lags = np.arange(0, 41)                          # τ = 0, 0.05, ..., 2.0
taus = lags * DT_LONG
emp_acov = np.array([acov(Xl, L) for L in lags])
fig, ax = plt.subplots()
ax.plot(taus, emp_acov, "o", ms=4, label="empirical autocovariance (one path)")
ax.plot(taus, V_STAT * np.exp(-THETA * taus), "C1", lw=2, label="$(\\sigma^2/2\\theta)\\,e^{-\\theta\\tau}$")
ax.set(xlabel="lag τ", ylabel="autocovariance", title=f"Autocovariance from one path of length T = {T_LONG:g}")
ax.legend(loc="upper right")
ins = ax.inset_axes([0.42, 0.35, 0.5, 0.35])
running = np.cumsum(Xl) / np.arange(1, n_long + 1)
ins.plot(np.arange(1, n_long + 1) * DT_LONG, running, lw=1)
ins.axhline(MU, color="C3", ls="--", lw=1)
ins.set(xscale="log", xlabel="t", ylim=(0.6, 1.8), title="running time average of $X_t$")
ins.title.set_fontsize(8)
plt.show()

경험 자기공분산이 $0.16e^{-2\tau}$ 를 따라 내려간다. 점들이 곡선 주위에서 **함께** 위아래로 흔들리는 것(인접 lag 의 추정 오차가 상관됨) 이 한 경로 추정의 특징이다. 삽입 그림: $3$ 에서 출발한 누적 시간평균이 $\mu=1$ 로 수렴하며, 초기 편차는 $\approx(x_0-\mu)/(\theta t)$ 로 사라진다.

### Step 5 — EM vs 정확 이산화 (같은 $\Delta t=0.1$)

07c 의 약오차를 구체적으로 본다. $\theta\Delta t=0.2$ 인 EM 은 계수 $0.8$ 의 AR(1) 이고 정상분산은 정리 3 의 $\sigma^2/(2\theta-\theta^2\Delta t)$; 같은 격자에서 `ou_exact` 는 계수 $e^{-0.2}$ 로 정확히 $0.16$ 이어야 한다. 평균은 두 방법 모두 편향이 없다 (EM 의 평균 재귀 $E[X_{k+1}]-\mu=(1-\theta\Delta t)(E[X_k]-\mu)$ 는 극한 $\mu$ 를 바꾸지 않는다).

In [ ]:
DT_EM, T_EM = 0.1, 5.0
n_em = int(round(T_EM / DT_EM))
_, X_em = euler_maruyama(lambda t, x: THETA * (MU - x), lambda t, x: SIGMA + 0 * x,
                         X0, T_EM, n_em, rng, n_paths=N_PATHS_EM)
_, X_ex = ou_exact(THETA, MU, SIGMA, X0, T_EM, n_em, rng, n_paths=N_PATHS_EM)   # 같은 Δt, 정확 이산화
Xe, Xx = X_em[:, -1], X_ex[:, -1]
V_EM = SIGMA**2 / (2 * THETA - THETA**2 * DT_EM)                 # 정리 3
z_true = (Xe.var(ddof=1) - V_STAT) / var_se(Xe)
print(f"AR(1) coefficient: EM 1-θΔt = {1 - THETA * DT_EM:.3f}  vs exact e^(-θΔt) = {np.exp(-THETA * DT_EM):.4f};"
      f"  EM variance transient (1-θΔt)^(2n) = {(1 - THETA * DT_EM) ** (2 * n_em):.1e}")
print("EM    mean X_5 :", mc_estimate(Xe), f" vs μ = {MU} (no bias)")
print(f"EM    var  X_5 : {Xe.var(ddof=1):.5f} ± {var_se(Xe):.5f}  vs EM formula σ²/(2θ-θ²Δt) = {V_EM:.6f}")
print(f"                 true σ²/2θ = {V_STAT} is {z_true:.1f} SE away  (excess {100 * (V_EM / V_STAT - 1):.1f}% predicted)")
print(f"exact var  X_5 : {Xx.var(ddof=1):.5f} ± {var_se(Xx):.5f}  vs σ²/2θ = {V_STAT}")

In [ ]:
grid = np.linspace(-0.8, 2.8, 400)
fig, ax = plt.subplots()
ax.hist(Xx, bins=60, density=True, alpha=0.5, label=f"exact sampler Δt={DT_EM} (var {Xx.var(ddof=1):.4f})")
ax.hist(Xe, bins=60, density=True, alpha=0.5, label=f"Euler–Maruyama Δt={DT_EM} (var {Xe.var(ddof=1):.4f})")
ax.plot(grid, norm.pdf(grid, MU, np.sqrt(V_STAT)), "k", lw=2, label=f"N({MU:g}, {V_STAT:g}) true stationary pdf")
ax.plot(grid, norm.pdf(grid, MU, np.sqrt(V_EM)), "k--", lw=1.5, label=f"N({MU:g}, {V_EM:.4f}) EM stationary pdf")
ax.set(xlabel="$X_5$", title="Stationary spread at t=5: exact sampler vs Euler–Maruyama (Δt=0.1)")
ax.legend(fontsize=8)
plt.show()

EM 히스토그램은 참 밀도보다 눈에 띄게 넓고, 정리 3 의 점선 밀도와는 맞는다. 07c 에서 "약오차 $O(\Delta t)$" 라 부른 것이 여기서는 정상분산의 $\frac{\theta\Delta t/2}{1-\theta\Delta t/2}=11\%$ 초과다. 정확 이산화는 같은 $\Delta t$ 에서 오차가 없다.

### Step 6 — 모수 추정 맛보기: AR(1) 회귀

정확 이산화 $X_{k+1}=aX_k+b+\varepsilon_k$ ($a=e^{-\theta\Delta t}$, $b=\mu(1-a)$, $\mathrm{Var}\,\varepsilon=\frac{\sigma^2}{2\theta}(1-a^2)$) 는 보통의 선형회귀다. Step 4 의 긴 경로에서 $\hat a,\hat b,s^2$ 를 구해 뒤집으면 $\hat\theta=-\log\hat a/\Delta t$, $\hat\mu=\hat b/(1-\hat a)$, $\hat\sigma^2=2\hat\theta s^2/(1-\hat a^2)$.

In [ ]:
theta_hat, mu_hat, sigma_hat, se_theta = ar1_fit(Xl, DT_LONG)
a_hat = np.exp(-theta_hat * DT_LONG)
print(f"n = {len(Xl) - 1} transitions, Δt = {DT_LONG}, T = {T_LONG:g}")
print(f"a_hat     = {a_hat:.5f}          vs e^(-θΔt) = {np.exp(-THETA * DT_LONG):.5f}")
print(f"theta_hat = {theta_hat:.4f} ± {se_theta:.4f}   vs θ = {THETA}   (rough SE sqrt(2θ/T) = {np.sqrt(2 * THETA / T_LONG):.4f})")
print(f"mu_hat    = {mu_hat:.4f}            vs μ = {MU}")
print(f"sigma_hat = {sigma_hat:.4f}            vs σ = {SIGMA}")

fig, ax = plt.subplots()
thin = max(1, len(Xl) // 5000)                  # 산점도는 최대 5000 점만
ax.plot(Xl[:-1:thin], Xl[1::thin], ".", ms=2, alpha=0.3, label="$(X_k, X_{k+1})$")
xx = np.array([Xl.min(), Xl.max()])
ax.plot(xx, a_hat * xx + mu_hat * (1 - a_hat), "C1", lw=2, label=f"fitted AR(1): slope {a_hat:.4f}")
ax.plot(xx, xx, "k:", lw=1, label="identity")
ax.set(xlabel="$X_k$", ylabel="$X_{k+1}$", title="Lag plot of one OU path with fitted AR(1) line (slope $e^{-\\theta\\Delta t}$)")
ax.legend()
plt.show()

$\hat\theta$ 의 SE 가 $\sqrt{2\theta/T}$ 정도라는 점에 주목하라 — $\Delta t$ 가 들어 있지 않다. 같은 $T$ 에서 $\Delta t$ 를 10배 촘촘히 찍어도 $\hat\theta$ 는 거의 정확해지지 않는다 (함정 5). $\hat\mu$, $\hat\sigma$ 의 SE 와 최우추정 전체는 후속 과정의 주제다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산한다 (정리 1 의 $m_t,v_t$, 정리 2 의 $\sigma^2/2\theta$ 와 $e^{-\theta\tau}$, 정리 3 의 EM 정상분산). $t=5$ 의 행들은 정상값과 비교한다 (전이 편향 $2e^{-10}$, $e^{-20}$ 은 SE 의 1/50 이하). "EM var vs TRUE" 행은 **일부러** 참값과 비교해 EM 의 편향이 판정에 잡히는지를 본다 — 그 행만 `investigate` 가 뜨는 것이 정답이다 (FAST 5 000 경로에서 $\approx5$ SE, full 20 000 경로에서 $\approx10$ SE; `assert` 셀의 마지막 줄에서 SE 배수를 확인하라).

In [ ]:
exact = {
    "mean_t05": ou_mean(t[k05]),                          # μ + (x0-μ) e^{-θ/2}
    "var_t05": ou_var(t[k05]),                            # σ²(1-e^{-θ})/(2θ)
    "mean_stat": MU,
    "var_stat": V_STAT,                                   # σ²/2θ
    "acov_lag05": V_STAT * np.exp(-THETA * 0.5),          # (σ²/2θ) e^{-θ·0.5}
    "tavg_mean": MU,
    "tavg_var": V_STAT,
    "em_var_dt01": V_EM,                                  # σ²/(2θ-θ²Δt)
    "em_var_vs_true": V_STAT,                             # 일부러 참값과 비교 (assert 안 함)
    "em_mean": MU,
    "exact_var_dt01": V_STAT,
    "theta_hat": THETA,
}
est = {                                                    # MCResult 또는 (값, SE)
    "mean_t05": mc_estimate(x05),
    "var_t05": (x05.var(ddof=1), var_se(x05)),
    "mean_stat": mc_estimate(x5),
    "var_stat": (x5.var(ddof=1), var_se(x5)),
    "acov_lag05": acov05,
    "tavg_mean": (tavg_mean, se_mean),
    "tavg_var": (tavg_var, se_var),
    "em_var_dt01": (Xe.var(ddof=1), var_se(Xe)),
    "em_var_vs_true": (Xe.var(ddof=1), var_se(Xe)),
    "em_mean": mc_estimate(Xe),
    "exact_var_dt01": (Xx.var(ddof=1), var_se(Xx)),
    "theta_hat": (theta_hat, se_theta),
}

In [ ]:
names = {
    "mean_t05": "E[X_0.5 | X_0=3]",
    "var_t05": "Var(X_0.5 | X_0=3)",
    "mean_stat": "E[X_5] (stationary mean)",
    "var_stat": "Var(X_5) (stationary variance)",
    "acov_lag05": "Cov(X_4.5, X_5)",
    "tavg_mean": f"time average of X, one path T={T_LONG:g} (batch-means SE)",
    "tavg_var": "time average of (X - X̄)², one path (batch-means SE)",
    "em_var_dt01": "Var(X_5^EM), Δt=0.1, vs EM formula σ²/(2θ-θ²Δt)",
    "em_var_vs_true": "Var(X_5^EM), Δt=0.1, vs TRUE σ²/2θ (bias expected!)",
    "em_mean": "E[X_5^EM], Δt=0.1",
    "exact_var_dt01": "Var(X_5^exact), Δt=0.1 (exact sampler)",
    "theta_hat": "θ̂ from AR(1) regression (one path)",
}
rows = []
for k, name in names.items():
    row = {"name": name, "predicted": predictions.get(k), "exact": exact[k]}
    if isinstance(est[k], tuple):
        row["estimate"], row["se"] = est[k]
    else:
        row["estimate"] = est[k]
    rows.append(row)
Markdown(compare_table(rows))

각 행에서 불일치가 뜻하는 것:
- **E[X_0.5], Var(X_0.5)**: 정리 1 의 $m_t,v_t$ 또는 `ou_exact` 의 전이 표본추출이 틀림 (예: 분산에 $2\theta$ 대신 $\theta$).
- **E[X_5], Var(X_5)**: 정상분포 $N(\mu,\sigma^2/2\theta)$ 가 틀렸거나 $t=5$ 가 아직 과도기 (상관시간 대비 짧음).
- **Cov(X_4.5, X_5)**: 자기공분산 $e^{-\theta\tau}$ 가 틀림 — $e^{-2\theta\tau}$ (분산의 감쇠율) 와 헷갈리면 여기서 걸린다.
- **time average (mean, var)**: 에르고딕성이 깨졌거나 (한 경로가 앙상블을 대표하지 못함) batch 길이가 상관시간보다 짧아 SE 를 과소평가함.
- **Var(X_5^EM) vs EM formula**: 정리 3 의 $\sigma^2/(2\theta-\theta^2\Delta t)$ 가 틀리거나 `euler_maruyama` 구현 오류.
- **Var(X_5^EM) vs TRUE**: 이 행은 `investigate` 가 **떠야 한다** (FAST $\approx5$ SE, full $\approx10$ SE) — 약오차 $0.0178$ 이 SE ($0.0036$ / $0.0018$) 보다 훨씬 크다는 뜻. `ok` 가 뜬다면 경로 수가 너무 적어 편향이 잡음에 묻힌 것이다 — 편향은 그대로인데 못 보는 것뿐이다 (2 000 경로면 SE $0.0056$, 딱 $3$ SE).
- **E[X_5^EM]**: EM 의 평균은 편향이 없어야 한다; 아니면 드리프트 계수 오류.
- **Var(X_5^exact), Δt=0.1**: 정확 이산화는 굵은 격자에서도 정확해야 한다; 아니면 `ou_exact` 의 잡음분산이 틀림.
- **θ̂**: AR(1) 회귀 공식이나 SE 공식 $\sqrt{(1-a^2)/n}/(a\Delta t)$ 가 틀림; 또는 경로가 정상 상태가 아님.

In [ ]:
assert_close(est["mean_t05"], exact["mean_t05"], k=4, name="mean t=0.5")
assert_close(est["var_t05"][0], exact["var_t05"], se=est["var_t05"][1], k=4, name="var t=0.5")
assert_close(est["mean_stat"], exact["mean_stat"], k=4, name="stationary mean")
assert_close(est["var_stat"][0], exact["var_stat"], se=est["var_stat"][1], k=4, name="stationary var")
assert_close(est["acov_lag05"], exact["acov_lag05"], k=4, name="acov lag 0.5")
assert_close(est["tavg_mean"][0], exact["tavg_mean"], se=est["tavg_mean"][1], k=4, name="time-average mean")
assert_close(est["tavg_var"][0], exact["tavg_var"], se=est["tavg_var"][1], k=4, name="time-average var")
assert_close(est["em_var_dt01"][0], exact["em_var_dt01"], se=est["em_var_dt01"][1], k=4, name="EM stationary var dt=0.1")
assert_close(est["em_mean"], exact["em_mean"], k=4, name="EM mean")
assert_close(est["exact_var_dt01"][0], exact["exact_var_dt01"], se=est["exact_var_dt01"][1], k=4, name="exact sampler var dt=0.1")
assert_close(est["theta_hat"][0], exact["theta_hat"], se=est["theta_hat"][1], k=4, name="theta_hat")
print("all assert_close passed (k=4)")
print(f"(deliberately NOT asserted) EM var vs true σ²/2θ: {z_true:.1f} SE — the O(Δt) weak error is real")

## 6. 연습문제

### E1 계산

$X_0=3$ (상수) 에서 출발한 OU ($\theta=2,\mu=1,\sigma=0.8$) 에 대해:
(a) 정리 1 의 공식으로 $\mathrm{Cov}(X_{0.5},X_1)$ 을 구하라. 정상 상태의 값 $0.16e^{-1}$ 과 왜 다른지 설명하라.
(b) 길이 $T=5000$ 의 한 경로가 주는 유효 독립 표본 수 $n_{\mathrm{eff}}$ 와, 그로부터 시간평균의 SE 를 계산하라 (Step 4 의 batch-means SE 와 비교).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $s=0.5,\ t=1$: $\mathrm{Cov}(X_s,X_t)=e^{-\theta(t-s)}v_s=e^{-1}\cdot\frac{0.64}{4}(1-e^{-2})=0.36788\times0.13835=0.050895$. 정상값 $0.16e^{-1}=0.058861$ 보다 작은 이유: $s=0.5$ 에서 분산이 아직 $v_{0.5}=0.1383<0.16$ 이기 때문이다 (상수에서 출발하면 초기에는 산포가 작다). 감쇠 인자 $e^{-\theta(t-s)}$ 는 같다 — 다른 것은 "출발점의 분산"뿐이다.

(b) 정리 2: $\mathrm{Var}(\bar X_T)\approx\frac{2}{\theta T}\cdot\frac{\sigma^2}{2\theta}$ 이므로 $n_{\mathrm{eff}}=\theta T/2=2\times5000/2=5000$, SE $=\sqrt{0.16/5000}=0.4/\sqrt{5000}=0.00566$ ($=\sigma/(\theta\sqrt T)$). 관측 수 $n=10^5$ 로 나눈 순진한 SE $0.4/\sqrt{10^5}=0.00126$ 의 4.5 배다. Step 4 의 batch-means SE 가 이 근처면 batch 길이가 충분했다는 뜻이다.

```python
s, tt = 0.5, 1.0
cov_st = np.exp(-THETA * (tt - s)) * ou_var(s)
print(f"Cov(X_0.5, X_1) = {cov_st:.6f}  vs stationary {V_STAT * np.exp(-THETA * (tt - s)):.6f}")
n_eff = THETA * 5000 / 2
print(f"n_eff = {n_eff:.0f}, SE = {np.sqrt(V_STAT / n_eff):.5f}  (naive {np.sqrt(V_STAT / 1e5):.5f})")
```

</details>

### E2 유도 후 시뮬레이션 검증

정확 이산화(AR(1) 계수 $e^{-\theta\Delta t}$) 의 정상분산이 **모든** $\Delta t$ 에서 $\sigma^2/2\theta$ 임을 고정점 방정식으로 보이고, EM 의 정상분산이 $\Delta t=0.5$ ($\theta\Delta t=1$, 계수 $a=0$) 에서 얼마인지 구하라. 그 다음 `ou_exact` 와 `euler_maruyama` 를 $\Delta t=0.5$ 로 $T=50$ 까지 돌려 $t=50$ 의 분산을 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

정확 이산화: $v=e^{-2\theta\Delta t}v+\frac{\sigma^2}{2\theta}(1-e^{-2\theta\Delta t})\Rightarrow v(1-e^{-2\theta\Delta t})=\frac{\sigma^2}{2\theta}(1-e^{-2\theta\Delta t})\Rightarrow v=\frac{\sigma^2}{2\theta}$ — $\Delta t$ 가 약분되어 사라진다. EM: $\sigma^2/(2\theta-\theta^2\Delta t)=0.64/(4-2)=0.32$, 참값의 **두 배**. $\theta\Delta t=1$ 이면 EM 계수 $a=0$ 이라 EM 경로는 매 스텝 과거를 완전히 잊는 iid 잡음 $\mu+\sigma\sqrt{\Delta t}\,\xi$ 가 된다 (분산 $0.64\times0.5=0.32$, 같은 답).

```python
dt2, T2 = 0.5, 50.0
n2 = int(round(T2 / dt2))
_, Xa = ou_exact(THETA, MU, SIGMA, X0, T2, n2, rng, n_paths=N_PATHS_EM)
_, Xb = euler_maruyama(lambda t, x: THETA * (MU - x), lambda t, x: SIGMA + 0 * x,
                       X0, T2, n2, rng, n_paths=N_PATHS_EM)
va, vb = Xa[:, -1].var(ddof=1), Xb[:, -1].var(ddof=1)
v_em2 = SIGMA**2 / (2 * THETA - THETA**2 * dt2)
print(f"exact sampler Δt=0.5: var {va:.4f} ± {var_se(Xa[:, -1]):.4f}  vs σ²/2θ = {V_STAT}")
print(f"EM            Δt=0.5: var {vb:.4f} ± {var_se(Xb[:, -1]):.4f}  vs σ²/(2θ-θ²Δt) = {v_em2}")
print("EM lag-1 autocorrelation at Δt=0.5:", np.corrcoef(Xb[:, -2], Xb[:, -1])[0, 1], " (coefficient 1-θΔt = 0)")
```

20 000 경로에서 대략 exact $0.159\pm0.002$, EM $0.32\pm0.003$ 이 나온다. EM 의 lag-1 자기상관은 $0$ 근처 — 정확 이산화의 $e^{-1}=0.37$ 과 전혀 다르다. 분산만이 아니라 **시간 구조**도 틀린 것이다.

</details>

### E3 가정을 깨보기

(a) $\theta\Delta t>2$: $\theta=2,\ \Delta t=1.1$ 로 EM 을 20 스텝 돌려 분산을 재고, 정리 3 증명의 부분합 공식 $\mathrm{Var}(X_k)=\sigma^2\Delta t\,\frac{a^{2k}-1}{a^2-1}$ ($a=1-\theta\Delta t=-1.2$, $k=20$) 과 비교하라.
(b) $\theta=-0.5$ (평균 **반발**): 정리 1 의 공식은 $\theta<0$ 에서도 그대로 성립한다. `ou_exact` 의 전이분산 식 $\frac{\sigma^2}{2\theta}(1-e^{-2\theta\Delta t})$ 이 $\theta<0$ 에서도 양수인지 확인하고, 왜 정상분포는 없는지 설명하라. 그 다음 `ou_exact` 와 EM ($\Delta t=0.01$) 으로 $T=5$ 까지 돌려 $\mathrm{Var}(X_5)$ 가 $\frac{\sigma^2}{2|\theta|}(e^{2|\theta|T}-1)$ 근처로 커짐을 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $a=1-2.2=-1.2$, $|a|>1$ 이므로 정상분포가 없다. 상수에서 출발한 EM 의 분산은 $\sigma^2\Delta t\sum_{j<k}a^{2j}=0.64\times1.1\times\frac{1.44^{20}-1}{0.44}\approx2350$ 이고, MC 도 `var_se` 로 잰 SE 안에서 그 근처다 (20 000 경로면 $\pm24$, FAST 5 000 경로면 $\pm47$). 경로는 부호를 바꿔 가며 지수적으로 커진다 — 참 OU 는 $\theta=2$ 로 얌전히 평균회귀하는데, 격자가 굵다는 이유만으로 EM 이 폭발한다.

(b) $\theta<0$ 이면 $1-e^{-2\theta\Delta t}<0$ 이고 $2\theta<0$ 이라 비율은 양수: $\frac{\sigma^2}{2|\theta|}(e^{2|\theta|\Delta t}-1)>0$. 즉 `ou_exact` 는 $\theta<0$ 에서도 **정확한 전이 표본추출기**다 (정리 1 의 유도에 $\theta>0$ 은 쓰이지 않았다). 없는 것은 정상분포다: $v_t=\frac{\sigma^2}{2|\theta|}(e^{2|\theta|t}-1)\to\infty$, 평균도 $\mu+(x_0-\mu)e^{|\theta|t}$ 로 도망간다. $T=5$: $v_5=0.64\,(e^5-1)=94.34$. EM 사슬($a=1.005$) 의 분산은 $\sigma^2\Delta t\frac{a^{1000}-1}{a^2-1}=92.9$ 로 $O(\Delta t)$ 만큼 작다 — 이번에는 EM 이 **과소**평가한다 (부호가 $\theta$ 에 따라 바뀐다).

```python
# (a)
dt3, k3 = 1.1, 20
a3 = 1 - THETA * dt3
_, Xc = euler_maruyama(lambda t, x: THETA * (MU - x), lambda t, x: SIGMA + 0 * x,
                       X0, dt3 * k3, k3, rng, n_paths=N_PATHS_EM)
v_formula = SIGMA**2 * dt3 * (a3 ** (2 * k3) - 1) / (a3**2 - 1)
print(f"(a) EM var after {k3} steps: {Xc[:, -1].var(ddof=1):.0f} ± {var_se(Xc[:, -1]):.0f}  vs formula {v_formula:.0f}")
# (b)
th_neg, dt4, T4 = -0.5, 0.01, 5.0
n4 = int(round(T4 / dt4))
_, Xd = ou_exact(th_neg, MU, SIGMA, X0, T4, n4, rng, n_paths=N_PATHS_EM)
_, Xf = euler_maruyama(lambda t, x: th_neg * (MU - x), lambda t, x: SIGMA + 0 * x,
                       X0, T4, n4, rng, n_paths=N_PATHS_EM)
v_true = SIGMA**2 / (2 * abs(th_neg)) * (np.exp(2 * abs(th_neg) * T4) - 1)
a4 = 1 - th_neg * dt4
v_em4 = SIGMA**2 * dt4 * (a4 ** (2 * n4) - 1) / (a4**2 - 1)
print(f"(b) exact sampler Var(X_5) = {Xd[:, -1].var(ddof=1):.1f} ± {var_se(Xd[:, -1]):.1f}  vs σ²(e^{{2|θ|T}}-1)/(2|θ|) = {v_true:.1f}")
print(f"    EM Δt=0.01     Var(X_5) = {Xf[:, -1].var(ddof=1):.1f} ± {var_se(Xf[:, -1]):.1f}  vs EM chain formula {v_em4:.1f}")
print(f"    mean X_5: {Xd[:, -1].mean():.2f} vs μ+(x0-μ)e^{{|θ|T}} = {MU + (X0 - MU) * np.exp(abs(th_neg) * T4):.2f}")
```

</details>

## 7. 정리

1. OU $dX=\theta(\mu-X)dt+\sigma dB$ 는 적분인자 $e^{\theta t}$ 로 풀린다: $X_t=\mu+(X_0-\mu)e^{-\theta t}+\sigma\int_0^te^{-\theta(t-s)}dB_s$ — 가우스, 평균 $\mu+(x-\mu)e^{-\theta t}$, 분산 $\frac{\sigma^2}{2\theta}(1-e^{-2\theta t})$ (등거리성).
2. 정상분포 $N(\mu,\sigma^2/2\theta)$, 자기공분산 $\frac{\sigma^2}{2\theta}e^{-\theta|\tau|}$; 앙상블과 한 긴 경로의 시간평균이 일치한다 (에르고딕). 시간평균의 SE 는 $s/\sqrt n$ 이 아니라 batch means 로 — 유효 표본 수 $\theta T/2$.
3. 정확 이산화는 AR(1) (계수 $e^{-\theta\Delta t}$) 로 어떤 $\Delta t$ 에서도 정확; EM 은 계수 $1-\theta\Delta t$ 라 정상분산이 $\sigma^2/(2\theta-\theta^2\Delta t)$ 로 커지고 ($\Delta t=0.1$ 에서 11 %), $\theta\Delta t\ge2$ 에서 발산한다.
4. AR(1) 회귀로 $\hat\theta=-\log\hat a/\Delta t$, $\hat\mu=\hat b/(1-\hat a)$, $\hat\sigma^2=2\hat\theta s^2/(1-\hat a^2)$; $\hat\theta$ 의 정보량은 관측 수가 아니라 $\theta T$ 에 비례한다.
5. 06b 의 가우스 과정 관점: BM 은 공분산 $\min(s,t)$, OU 는 $\frac{\sigma^2}{2\theta}e^{-\theta|s-t|}$ — 공분산 함수가 과정을 규정한다.

**Trap 카드.** Q: OU 의 정상분산은 $\sigma^2/\theta$ 인가?
→ A: 아니다, $\sigma^2/(2\theta)$. 등거리성으로 $\mathrm{Var}=\sigma^2\int_0^\infty e^{-2\theta s}ds=\sigma^2/(2\theta)$ — 지수 안의 $2$ 가 분모로 내려온다. (EM 은 이것을 $\sigma^2/(2\theta-\theta^2\Delta t)$ 로 부풀린다.)

**다음 노트북: 07e (기하 브라운 운동과 블랙–숄즈).** 선형 SDE 의 두 번째 대표인 GBM 으로 돌아가 Itô 공식으로 해를 구하고, 위험중립 기댓값으로 블랙–숄즈 공식을 유도한다. 오늘 본 "기댓값 = 전이분포의 적분" 연결이 파인만–카츠(Feynman–Kac) 정리로 일반화되어 기댓값이 PDE 의 해가 된다.

`log/progress.md` 한 줄 템플릿:
`- [ ] 07d 오른슈타인-울렌벡 과정 — 날짜: ____ / 소요: ___분 / 예측 적중: _/6 / 메모: ____`